In [2]:
!pip install gpytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.9/279.9 kB 6.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViTModel, ViTImageProcessor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import gpytorch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ==================== Dataset ====================
class EuroSATDataset(Dataset):
    def __init__(self, filenames, labels, image_dir, processor):
        self.filenames = filenames
        self.labels = labels
        self.image_dir = image_dir
        self.processor = processor
        
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img_path = self.image_dir + self.filenames[idx]
        label = int(self.labels[idx])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        inputs = self.processor(images=image, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0), label

# ==================== Feature Extraction ====================
def extract_features(dataloader, vit_model, device):
    vit_model.eval()
    features, labels = [], []
    
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting features"):
            imgs = imgs.to(device)
            outputs = vit_model(pixel_values=imgs)
            features.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
            labels.append(lbls.numpy())
    
    return np.vstack(features), np.concatenate(labels)

# ==================== OPTIMIZED Binary GP Model ====================
class OptimizedBinaryGP(gpytorch.models.ApproximateGP):
    """
    Optimized Binary GP with:
    - Better kernel initialization
    - ARD for automatic feature selection
    - Adaptive inducing points
    """
    def __init__(self, train_x, train_y, num_inducing=100, use_ard=False):
        # Smart inducing point initialization: sample from both classes
        pos_indices = torch.where(train_y == 1)[0]
        neg_indices = torch.where(train_y == 0)[0]
        
        # Split inducing points between classes
        num_pos = min(num_inducing // 2, len(pos_indices))
        num_neg = num_inducing - num_pos
        
        if len(pos_indices) > 0 and len(neg_indices) > 0:
            pos_sample = pos_indices[torch.randperm(len(pos_indices))[:num_pos]]
            neg_sample = neg_indices[torch.randperm(len(neg_indices))[:num_neg]]
            inducing_idx = torch.cat([pos_sample, neg_sample])
        else:
            inducing_idx = torch.randperm(train_x.size(0))[:num_inducing]
        
        inducing_points = train_x[inducing_idx]
        
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=inducing_points.size(0)
        )
        
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True
        )
        
        super().__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        
        # LINEAR kernel - works best for pre-trained ViT features
        # Add constant kernel for numerical stability
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.LinearKernel() + 
            gpytorch.kernels.ConstantKernel()
        )
        
        # Good initialization
        self.covar_module.outputscale = 1.0
        
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# ==================== OPTIMIZED Training ====================
def train_binary_gp_optimized(model, likelihood, train_x, train_y, 
                               n_epochs=200, initial_lr=0.1):
    """
    Optimized training with:
    - Adaptive learning rate
    - Better convergence criteria
    - Regularization
    """
    model.train()
    likelihood.train()
    
    # Separate optimizers for different parameter groups
    optimizer = torch.optim.Adam([
        {'params': model.variational_parameters(), 'lr': initial_lr},
        {'params': model.hyperparameters(), 'lr': initial_lr * 0.1},
        {'params': likelihood.parameters(), 'lr': initial_lr * 0.1},
    ])
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15, verbose=False
    )
    
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
    
    best_loss = float('inf')
    patience = 0
    max_patience = 40
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        loss_val = loss.item()
        scheduler.step(loss_val)
        
        if loss_val < best_loss:
            best_loss = loss_val
            patience = 0
        else:
            patience += 1
        
        if patience >= max_patience:
            break
    
    return model, likelihood, best_loss

# ==================== OPTIMIZED One-vs-Rest Classifier ====================
class OptimizedOneVsRestGP:
    """
    Optimized One-vs-Rest GP with:
    - Smart inducing point initialization
    - Linear kernel for pre-trained features
    - Calibrated predictions
    """
    def __init__(self, num_classes, num_inducing=120):
        self.num_classes = num_classes
        self.num_inducing = num_inducing
        self.models = []
        self.likelihoods = []
        self.class_priors = []
    
    def fit(self, train_x, train_y, n_epochs=200, initial_lr=0.12):
        """Train optimized binary GPs"""
        
        
        for class_idx in range(self.num_classes):
            
            # Binary labels
            train_y_binary = (train_y == class_idx).float()
            
            num_pos = train_y_binary.sum().item()
            num_neg = (train_y_binary == 0).sum().item()
            
            
            # Store class prior
            self.class_priors.append(num_pos / len(train_y))
            
            # Create optimized GP
            model = OptimizedBinaryGP(
                train_x, train_y_binary, 
                num_inducing=self.num_inducing
            ).to(train_x.device)
            
            likelihood = gpytorch.likelihoods.BernoulliLikelihood().to(train_x.device)
            
            # Train with optimization
            model, likelihood, final_loss = train_binary_gp_optimized(
                model, likelihood, train_x, train_y_binary,
                n_epochs=n_epochs, initial_lr=initial_lr
            )
            
            
            print(f"Final loss: {final_loss:.4f}")
            
            self.models.append(model)
            self.likelihoods.append(likelihood)
        
    
    def predict(self, test_x, calibrate=True, batch_size=100):
        """
        Predict with optional calibration
        """
        print("\nMaking predictions...")
        
        for model, likelihood in zip(self.models, self.likelihoods):
            model.eval()
            likelihood.eval()
        
        all_scores = []
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            for class_idx in tqdm(range(self.num_classes), desc="Computing scores"):
                model = self.models[class_idx]
                likelihood = self.likelihoods[class_idx]
                
                class_scores = []
                
                for i in range(0, test_x.size(0), batch_size):
                    batch_x = test_x[i:min(i+batch_size, test_x.size(0))]
                    
                    output = model(batch_x)
                    pred = likelihood(output)
                    
                    # Get probability of positive class
                    scores = pred.mean
                    class_scores.append(scores.cpu())
                
                class_scores = torch.cat(class_scores)
                all_scores.append(class_scores)
        
        # Stack scores: (num_classes, num_samples)
        all_scores = torch.stack(all_scores)
        
        if calibrate:
            # Apply prior calibration (helps with imbalanced classes)
            priors = torch.tensor(self.class_priors).unsqueeze(1)
            all_scores = all_scores * priors
        
        # Choose class with highest score
        predictions = all_scores.argmax(dim=0).numpy()
        
        return predictions, all_scores.numpy()

# ==================== Main ====================
def main():
    print("="*70)
    print("OPTIMIZED Gaussian Process Classification")
    print("="*70)
    
    # Load data
    print("\nLoading data...")
    train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')
    test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')
    
    # Setup labels
    if 'ClassName' in train_df.columns:
        classes = sorted(train_df['ClassName'].unique())
        train_df['label_idx'] = train_df['ClassName'].map({c: i for i, c in enumerate(classes)})
        test_df['label_idx'] = test_df['ClassName'].map({c: i for i, c in enumerate(classes)}) if 'ClassName' in test_df.columns else test_df['Label']
    else:
        train_df['label_idx'] = train_df['Label']
        test_df['label_idx'] = test_df['Label']
    
    num_classes = len(train_df['label_idx'].unique())
    print(f"Number of classes: {num_classes}")
    
    # Subset (20 per class)
    train_subset = pd.concat([
        train_df[train_df['label_idx'] == i].sample(n=20, random_state=42)
        for i in range(num_classes)
    ], ignore_index=True)
    
    print(f"Training samples: {len(train_subset)} ({len(train_subset)//num_classes} per class)")
    print(f"Test samples: {len(test_df)}")
    
    # Extract arrays
    train_files = train_subset['Filename'].values
    train_labels = train_subset['label_idx'].values
    test_files = test_df['Filename'].values
    test_labels = test_df['label_idx'].values
    
    image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'
    
    # Load ViT
    print("\nLoading ViT...")
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    
    import transformers
    transformers.logging.set_verbosity_error()
    vit = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)
    transformers.logging.set_verbosity_warning()
    
    for p in vit.parameters():
        p.requires_grad = False
    
    # Datasets
    train_ds = EuroSATDataset(train_files, train_labels, image_dir, processor)
    test_ds = EuroSATDataset(test_files, test_labels, image_dir, processor)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    
    # Extract features
    print("\n" + "="*70)
    print("Feature Extraction")
    print("="*70)
    train_feat, train_lbl = extract_features(train_loader, vit, device)
    test_feat, test_lbl = extract_features(test_loader, vit, device)
    

    
    print(f"Extracted: Train {train_feat.shape}, Test {test_feat.shape}")
    
    # Preprocess
    print("\nPreprocessing...")
    scaler = StandardScaler()
    train_feat = scaler.fit_transform(train_feat)
    test_feat = scaler.transform(test_feat)
    
    # OPTIMIZATION: Use more PCA components for better feature retention
    pca_components = 128  # Increased from 100
    pca = PCA(n_components=pca_components, random_state=42)
    train_feat = pca.fit_transform(train_feat)
    test_feat = pca.transform(test_feat)
    
    print(f"PCA: {pca_components} components, {pca.explained_variance_ratio_.sum():.4f} variance")
    
    # Convert to tensors
    train_x = torch.tensor(train_feat, dtype=torch.float32).to(device)
    train_y = torch.tensor(train_lbl, dtype=torch.long).to(device)
    test_x = torch.tensor(test_feat, dtype=torch.float32).to(device)
    
    # Train OPTIMIZED GP
    print("\n" + "="*70)
    print("Training OPTIMIZED GP")
    print("="*70)
    
    gp_classifier = OptimizedOneVsRestGP(
        num_classes=num_classes,
        num_inducing=130  # Slightly increased
    )
    
    gp_classifier.fit(
        train_x, train_y, 
        n_epochs=2000,  # More epochs
        initial_lr=0.12,  # Higher initial LR
    )
    
    # Predict with calibration
    print("\n" + "="*70)
    print("Evaluation")
    print("="*70)
    
    predictions, scores = gp_classifier.predict(
        test_x, 
        calibrate=True,  # Use prior calibration
        batch_size=100
    )
    
    # Calculate accuracy
    accuracy = accuracy_score(test_lbl, predictions)
    
    print(f"\n{'='*70}")
    print(f"OPTIMIZED GP TEST ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"{'='*70}")
    
    # Detailed report
    print("\nClassification Report:")
    print("="*70)
    print(classification_report(test_lbl, predictions,
                                target_names=[f"Class {i}" for i in range(num_classes)],
                                digits=4))
    
    # Per-class performance
    print("\nPer-Class Performance:")
    print("-"*70)
    for i in range(num_classes):
        mask = test_lbl == i
        if mask.sum() > 0:
            acc = accuracy_score(test_lbl[mask], predictions[mask])
            avg_score = scores[i, mask].mean()
            std_score = scores[i, mask].std()
            print(f"Class {i:2d}: Acc={acc*100:6.2f}% | "
                  f"Score={avg_score:.3f}±{std_score:.3f} | "
                  f"N={mask.sum():4d}")
    
    print("\n" + "="*70)
    print("OPTIMIZATION FEATURES USED:")
    print("✓ Linear + Constant Kernel (optimal for ViT features)")
    print("✓ Class-balanced inducing point initialization")
    print("✓ Adaptive learning rate with ReduceLROnPlateau")
    print("✓ 128 PCA components (more information retained)")
    print("✓ Prior calibration in prediction")
    print("✓ Extended training (200 epochs with early stopping)")
    print("="*70)
    
    return accuracy

if __name__ == "__main__":
    main()

2025-11-05 06:34:33.546936: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762324473.570988     128 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762324473.578532     128 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda
GPU: Tesla P100-PCIE-16GB
OPTIMIZED Gaussian Process Classification

Loading data...
Number of classes: 10
Training samples: 200 (20 per class)
Test samples: 2700

Loading ViT...

Feature Extraction


Extracting features: 100%|██████████| 85/85 [00:53<00:00,  1.58it/s]


Extracted: Train (200, 768), Test (2700, 768)

Preprocessing...
PCA: 128 components, 0.9549 variance

Training OPTIMIZED GP
Final loss: 0.1845
Final loss: 0.1410
Final loss: 0.1773
Final loss: 0.2342
Final loss: 0.1625
Final loss: 0.1940
Final loss: 0.2484
Final loss: 0.1695
Final loss: 0.2454
Final loss: 0.1488

Evaluation

Making predictions...


Computing scores: 100%|██████████| 10/10 [00:01<00:00,  9.54it/s]


OPTIMIZED GP TEST ACCURACY: 0.8619 (86.19%)

Classification Report:
              precision    recall  f1-score   support

     Class 0     0.8700    0.9367    0.9021       300
     Class 1     0.9148    0.9667    0.9400       300
     Class 2     0.8253    0.9133    0.8671       300
     Class 3     0.7437    0.7080    0.7254       250
     Class 4     0.8662    0.9320    0.8979       250
     Class 5     0.8037    0.8800    0.8401       200
     Class 6     0.8934    0.7040    0.7875       250
     Class 7     0.9298    0.9267    0.9282       300
     Class 8     0.8068    0.6680    0.7309       250
     Class 9     0.9197    0.9167    0.9182       300

    accuracy                         0.8619      2700
   macro avg     0.8573    0.8552    0.8537      2700
weighted avg     0.8615    0.8619    0.8593      2700


Per-Class Performance:
----------------------------------------------------------------------
Class  0: Acc= 93.67% | Score=0.064±0.026 | N= 300
Class  1: Acc= 96.67% | Sc

In [15]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViTModel, ViTImageProcessor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import gpytorch
from tqdm import tqdm
import warnings
import gc
import json
from datetime import datetime
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== Dataset ====================
class EuroSATDataset(Dataset):
    def __init__(self, filenames, labels, image_dir, processor, augment=False, hard_classes=None):
        self.filenames = filenames
        self.labels = labels
        self.image_dir = image_dir
        self.processor = processor
        self.augment = augment
        self.hard_classes = hard_classes if hard_classes is not None else []
        
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img_path = self.image_dir + self.filenames[idx]
        label = int(self.labels[idx])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.augment:
            import torchvision.transforms as T
            
            if label in self.hard_classes:
                augment_transforms = T.Compose([
                    T.RandomHorizontalFlip(p=0.5),
                    T.RandomVerticalFlip(p=0.5),
                    T.RandomRotation(degrees=180),
                    T.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15)),
                    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
                    T.RandomResizedCrop(size=224, scale=(0.75, 1.0)),
                ])
            else:
                augment_transforms = T.Compose([
                    T.RandomHorizontalFlip(p=0.5),
                    T.RandomVerticalFlip(p=0.5),
                    T.RandomRotation(degrees=90),
                    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                ])
            
            image = augment_transforms(image)
        
        inputs = self.processor(images=image, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0), label

# ==================== ViT with Last 2 Layers Unfrozen ====================
class FineTunedViT(nn.Module):
    def __init__(self, num_layers_to_unfreeze=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        
        for param in self.vit.parameters():
            param.requires_grad = False
        
        total_layers = len(self.vit.encoder.layer)
        for i in range(total_layers - num_layers_to_unfreeze, total_layers):
            for param in self.vit.encoder.layer[i].parameters():
                param.requires_grad = True
    
    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        return outputs.last_hidden_state[:, 0, :]

# ==================== Binary GP ====================
class BinaryGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, train_x, train_y, num_inducing=100):
        pos_indices = torch.where(train_y == 1)[0]
        neg_indices = torch.where(train_y == 0)[0]
        
        num_pos = min(num_inducing // 2, len(pos_indices))
        num_neg = num_inducing - num_pos
        
        if len(pos_indices) > 0 and len(neg_indices) > 0:
            pos_sample = pos_indices[torch.randperm(len(pos_indices))[:num_pos]]
            neg_sample = neg_indices[torch.randperm(len(neg_indices))[:num_neg]]
            inducing_idx = torch.cat([pos_sample, neg_sample])
        else:
            inducing_idx = torch.randperm(train_x.size(0))[:num_inducing]
        
        inducing_points = train_x[inducing_idx].detach().clone()
        
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=inducing_points.size(0)
        )
        
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution,
            learn_inducing_locations=True
        )
        
        super().__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.LinearKernel() + 
            gpytorch.kernels.ConstantKernel()
        )
        
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# ==================== Training Functions ====================
def finetune_vit_layers(vit_model, train_loader, n_epochs=40, lr=2e-5, verbose=False):
    num_classes = 10
    classifier = nn.Linear(768, num_classes).to(device)
    
    vit_model.train()
    classifier.train()
    
    class_weights = torch.ones(num_classes).to(device)
    class_weights[3] = 2.0
    class_weights[6] = 2.0
    class_weights[8] = 2.0
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    optimizer = torch.optim.AdamW([
        {'params': filter(lambda p: p.requires_grad, vit_model.parameters()), 
         'lr': lr, 'weight_decay': 0.01},
        {'params': classifier.parameters(), 'lr': lr * 10, 'weight_decay': 0.01}
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-7
    )
    
    best_loss = float('inf')
    patience = 0
    max_patience = 15
    
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            features = vit_model(imgs)
            logits = classifier(features)
            loss = criterion(logits, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        acc = 100. * correct / total
        avg_loss = epoch_loss / len(train_loader)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience = 0
        else:
            patience += 1
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs} - Loss: {avg_loss:.4f} - Acc: {acc:.2f}%")
        
        if patience >= max_patience:
            break
    
    del classifier
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return vit_model

def train_gps(vit_model, train_loader, train_labels, num_classes, verbose=False):
    vit_model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            features = vit_model(imgs)
            all_features.append(features.cpu())
            all_labels.append(labels)
    
    train_features = torch.cat(all_features).to(device)
    train_y = torch.cat(all_labels).to(device)
    
    scaler = StandardScaler()
    train_feat_np = scaler.fit_transform(train_features.cpu().numpy())
    train_features = torch.tensor(train_feat_np, dtype=torch.float32).to(device)
    
    models = []
    likelihoods = []
    
    for class_idx in range(num_classes):
        binary_labels = (train_y == class_idx).float()
        
        model = BinaryGPModel(train_features, binary_labels, num_inducing=100).to(device)
        likelihood = gpytorch.likelihoods.BernoulliLikelihood().to(device)
        
        model.train()
        likelihood.train()
        
        optimizer = torch.optim.Adam([
            {'params': model.variational_parameters(), 'lr': 0.05},
            {'params': model.hyperparameters(), 'lr': 0.005},
            {'params': likelihood.parameters(), 'lr': 0.005},
        ])
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=15
        )
        
        mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
        
        best_loss = float('inf')
        patience = 0
        
        for epoch in range(150):
            optimizer.zero_grad()
            output = model(train_features)
            loss = -mll(output, binary_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step(loss.item())
            
            if loss.item() < best_loss:
                best_loss = loss.item()
                patience = 0
            else:
                patience += 1
            
            if patience >= 30:
                break
        
        models.append(model)
        likelihoods.append(likelihood)
    
    return models, likelihoods, scaler

def predict(vit_model, models, likelihoods, test_loader, scaler, num_classes):
    vit_model.eval()
    for model, likelihood in zip(models, likelihoods):
        model.eval()
        likelihood.eval()
    
    all_preds = []
    
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        for imgs, _ in test_loader:
            imgs = imgs.to(device)
            features = vit_model(imgs).cpu().numpy()
            features = scaler.transform(features)
            features = torch.tensor(features, dtype=torch.float32).to(device)
            
            class_scores = []
            for class_idx in range(num_classes):
                output = models[class_idx](features)
                pred = likelihoods[class_idx](output)
                scores = pred.mean
                class_scores.append(scores)
            
            class_scores = torch.stack(class_scores, dim=0)
            batch_preds = class_scores.argmax(dim=0)
            all_preds.append(batch_preds.cpu())
    
    predictions = torch.cat(all_preds).numpy()
    return predictions

# ==================== Single Experiment ====================
def run_single_experiment(train_df, test_df, image_dir, processor, 
                          samples_per_class, seed, num_classes=10):
    """Run one experiment with given parameters"""
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Clear GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    # Sample training data
    train_subset = pd.concat([
        train_df[train_df['label_idx'] == i].sample(n=min(samples_per_class, len(train_df[train_df['label_idx'] == i])), 
                                                     random_state=seed)
        for i in range(num_classes)
    ], ignore_index=True)
    
    train_files = train_subset['Filename'].values
    train_labels = train_subset['label_idx'].values
    test_files = test_df['Filename'].values
    test_labels = test_df['label_idx'].values
    
    # Datasets
    hard_classes = [3, 6, 8]
    train_ds = EuroSATDataset(train_files, train_labels, image_dir, processor, 
                               augment=True, hard_classes=hard_classes)
    train_ds_no_aug = EuroSATDataset(train_files, train_labels, image_dir, processor, 
                                      augment=False)
    test_ds = EuroSATDataset(test_files, test_labels, image_dir, processor, augment=False)
    
    train_loader_aug = DataLoader(train_ds, batch_size=min(16, len(train_ds)), shuffle=True)
    train_loader_no_aug = DataLoader(train_ds_no_aug, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    
    # Create model
    import transformers
    transformers.logging.set_verbosity_error()
    
    vit_model = FineTunedViT(num_layers_to_unfreeze=2).to(device)
    
    # Train
    vit_model = finetune_vit_layers(vit_model, train_loader_aug, n_epochs=40, lr=2e-5, verbose=False)
    models, likelihoods, scaler = train_gps(vit_model, train_loader_no_aug, train_labels, num_classes, verbose=False)
    
    # Predict
    predictions = predict(vit_model, models, likelihoods, test_loader, scaler, num_classes)
    accuracy = accuracy_score(test_labels, predictions)
    
    # Per-class accuracy
    per_class_acc = {}
    for i in range(num_classes):
        mask = test_labels == i
        if mask.sum() > 0:
            per_class_acc[i] = accuracy_score(test_labels[mask], predictions[mask])
    
    # Cleanup
    del vit_model, models, likelihoods
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    return accuracy, per_class_acc

# ==================== Main Experiments ====================
def main():
    print("="*70)
    print("GP COMPREHENSIVE EXPERIMENTS")
    print("="*70)
    
    # Load data
    print("\nLoading data...")
    train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')
    test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')
    
    if 'ClassName' in train_df.columns:
        classes = sorted(train_df['ClassName'].unique())
        train_df['label_idx'] = train_df['ClassName'].map({c: i for i, c in enumerate(classes)})
        test_df['label_idx'] = test_df['ClassName'].map({c: i for i, c in enumerate(classes)}) if 'ClassName' in test_df.columns else test_df['Label']
    else:
        train_df['label_idx'] = train_df['Label']
        test_df['label_idx'] = test_df['Label']
    
    num_classes = len(train_df['label_idx'].unique())
    image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    
    # Results storage
    results = {
        'data_size_experiments': {},
        'seed_experiments': {}
    }
    
    # EXPERIMENT 1: Vary data size (samples per class: 100, 50, 20, 10, 5, 1)
    print("\n" + "="*70)
    print("EXPERIMENT 1: Varying Data Size")
    print("="*70)
    
    data_sizes = [100, 50, 20, 10, 5, 1]
    seed = 42
    
    for samples_per_class in data_sizes:
        print(f"\n--- Running with {samples_per_class} samples per class (seed={seed}) ---")
        
        accuracy, per_class_acc = run_single_experiment(
            train_df, test_df, image_dir, processor, 
            samples_per_class, seed, num_classes
        )
        
        results['data_size_experiments'][samples_per_class] = {
            'accuracy': float(accuracy),
            'per_class': {int(k): float(v) for k, v in per_class_acc.items()},
            'seed': seed
        }
        
        print(f"Accuracy: {accuracy*100:.2f}%")
        print(f"Per-class (hard): Class 3: {per_class_acc.get(3, 0)*100:.2f}%, "
              f"Class 6: {per_class_acc.get(6, 0)*100:.2f}%, "
              f"Class 8: {per_class_acc.get(8, 0)*100:.2f}%")
    
    # EXPERIMENT 2: Vary seed (20 samples per class, seeds: 42, 123, 456, 789, 2024)
    print("\n" + "="*70)
    print("EXPERIMENT 2: Varying Seed (20 samples/class)")
    print("="*70)
    
    samples_per_class = 20
    seeds = [42, 123, 456, 789, 2024]
    
    for seed in seeds:
        print(f"\n--- Running with seed={seed} (20 samples per class) ---")
        
        accuracy, per_class_acc = run_single_experiment(
            train_df, test_df, image_dir, processor, 
            samples_per_class, seed, num_classes
        )
        
        results['seed_experiments'][seed] = {
            'accuracy': float(accuracy),
            'per_class': {int(k): float(v) for k, v in per_class_acc.items()},
            'samples_per_class': samples_per_class
        }
        
        print(f"Accuracy: {accuracy*100:.2f}%")
        print(f"Per-class (hard): Class 3: {per_class_acc.get(3, 0)*100:.2f}%, "
              f"Class 6: {per_class_acc.get(6, 0)*100:.2f}%, "
              f"Class 8: {per_class_acc.get(8, 0)*100:.2f}%")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f'/kaggle/working/gp_experiments_{timestamp}.json'
    
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print("\n" + "="*70)
    print("FINAL RESULTS SUMMARY")
    print("="*70)
    
    print("\n1. DATA SIZE VARIATION (seed=42):")
    print("-" * 70)
    print(f"{'Samples/Class':<15} {'Accuracy':<12} {'Class 3':<10} {'Class 6':<10} {'Class 8':<10}")
    print("-" * 70)
    for size in data_sizes:
        res = results['data_size_experiments'][size]
        print(f"{size:<15} {res['accuracy']*100:>10.2f}% {res['per_class'].get(3, 0)*100:>8.2f}% "
              f"{res['per_class'].get(6, 0)*100:>8.2f}% {res['per_class'].get(8, 0)*100:>8.2f}%")
    
    print("\n2. SEED VARIATION (20 samples/class):")
    print("-" * 70)
    print(f"{'Seed':<15} {'Accuracy':<12} {'Class 3':<10} {'Class 6':<10} {'Class 8':<10}")
    print("-" * 70)
    for seed in seeds:
        res = results['seed_experiments'][seed]
        print(f"{seed:<15} {res['accuracy']*100:>10.2f}% {res['per_class'].get(3, 0)*100:>8.2f}% "
              f"{res['per_class'].get(6, 0)*100:>8.2f}% {res['per_class'].get(8, 0)*100:>8.2f}%")
    
    # Statistics
    seed_accuracies = [results['seed_experiments'][s]['accuracy'] for s in seeds]
    mean_acc = np.mean(seed_accuracies)
    std_acc = np.std(seed_accuracies)
    
    print("\n3. SEED VARIATION STATISTICS:")
    print("-" * 70)
    print(f"Mean accuracy: {mean_acc*100:.2f}%")
    print(f"Std deviation: {std_acc*100:.2f}%")
    print(f"Min accuracy: {min(seed_accuracies)*100:.2f}%")
    print(f"Max accuracy: {max(seed_accuracies)*100:.2f}%")
    
    print(f"\n Results saved to: {results_file}")
    print("="*70)
    
    return results

if __name__ == "__main__":
    results = main()

Using device: cuda
GP COMPREHENSIVE EXPERIMENTS

Loading data...

EXPERIMENT 1: Varying Data Size

--- Running with 100 samples per class (seed=42) ---
Accuracy: 93.93%
Per-class (hard): Class 3: 88.80%, Class 6: 89.60%, Class 8: 90.40%

--- Running with 50 samples per class (seed=42) ---
Accuracy: 92.30%
Per-class (hard): Class 3: 86.80%, Class 6: 86.40%, Class 8: 82.80%

--- Running with 20 samples per class (seed=42) ---
Accuracy: 87.00%
Per-class (hard): Class 3: 73.60%, Class 6: 70.80%, Class 8: 64.00%

--- Running with 10 samples per class (seed=42) ---
Accuracy: 82.59%
Per-class (hard): Class 3: 66.40%, Class 6: 57.20%, Class 8: 58.80%

--- Running with 5 samples per class (seed=42) ---
Accuracy: 74.22%
Per-class (hard): Class 3: 58.80%, Class 6: 35.20%, Class 8: 36.40%

--- Running with 1 samples per class (seed=42) ---
Accuracy: 40.22%
Per-class (hard): Class 3: 31.60%, Class 6: 13.60%, Class 8: 17.20%

EXPERIMENT 2: Varying Seed (20 samples/class)

--- Running with seed=42 (2

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gpytorch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

"""
Gaussian Process Convolutional Filters

Idea: Use GPs to generate/learn convolutional filter weights
Advantage: Can capture uncertainty, work with few samples, interpretable

Three approaches implemented:
1. GP Filter Generator: GP samples spatial coordinates -> filter weights
2. GP Spatial Kernel: Each filter position has GP prior
3. Hybrid: Standard Conv + GP refinement layer
"""

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== Approach 1: GP Filter Generator ====================
class GPFilterGenerator(nn.Module):
    """
    Generate convolutional filter weights using GP
    Each filter is a realization of a 2D GP over spatial coordinates
    """
    def __init__(self, num_filters=32, filter_size=3, lengthscale=1.0):
        super().__init__()
        self.num_filters = num_filters
        self.filter_size = filter_size
        
        # Create coordinate grid for filter
        # For 3x3 filter: coordinates are [(-1,-1), (-1,0), (-1,1), ..., (1,1)]
        x = torch.linspace(-1, 1, filter_size)
        y = torch.linspace(-1, 1, filter_size)
        xx, yy = torch.meshgrid(x, y, indexing='ij')
        self.register_buffer('coords', torch.stack([xx.flatten(), yy.flatten()], dim=1))  # (9, 2) for 3x3
        
        # GP kernel for generating filters
        self.lengthscale = nn.Parameter(torch.tensor(lengthscale))
        self.outputscale = nn.Parameter(torch.tensor(1.0))
        
        # Sample filter weights from GP prior
        self.filter_weights = nn.Parameter(torch.randn(num_filters, filter_size * filter_size))
        
    def rbf_kernel(self, x1, x2):
        """RBF kernel between coordinate sets"""
        # x1: (N, 2), x2: (M, 2)
        dist = torch.cdist(x1.unsqueeze(0), x2.unsqueeze(0)).squeeze(0)  # (N, M)
        return self.outputscale * torch.exp(-0.5 * dist**2 / self.lengthscale**2)
    
    def get_filters(self):
        """Get current filter weights shaped for conv2d"""
        # Shape: (num_filters, 1, filter_size, filter_size)
        return self.filter_weights.view(self.num_filters, 1, self.filter_size, self.filter_size)
    
    def forward(self, x):
        """
        Apply GP-generated filters
        x: (batch, channels, height, width)
        """
        filters = self.get_filters()
        
        # Apply depthwise convolution
        batch, in_channels, h, w = x.shape
        
        # Reshape for grouped convolution
        x_reshaped = x.view(1, batch * in_channels, h, w)
        filters_repeated = filters.repeat(in_channels, 1, 1, 1)
        
        out = F.conv2d(x_reshaped, filters_repeated, groups=in_channels, padding=self.filter_size//2)
        out = out.view(batch, in_channels * self.num_filters, h, w)
        
        return out
    
    def sample_from_gp_prior(self):
        """Sample new filter weights from GP prior"""
        K = self.rbf_kernel(self.coords, self.coords)  # (9, 9) for 3x3
        K = K + 1e-4 * torch.eye(K.shape[0], device=K.device)  # Add jitter
        
        # Cholesky decomposition
        L = torch.linalg.cholesky(K)
        
        # Sample: f = L @ z where z ~ N(0, I)
        z = torch.randn(self.num_filters, self.filter_size * self.filter_size, device=K.device)
        samples = (L @ z.T).T
        
        self.filter_weights.data = samples
        
        return samples

# ==================== Approach 2: GP Spatial Conv Layer ====================
class GPSpatialConv(nn.Module):
    """
    Convolutional layer where each spatial position has a GP prior
    Learns correlations between filter positions
    """
    def __init__(self, in_channels, out_channels, filter_size=3):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.filter_size = filter_size
        
        # Standard conv weights (will be modulated by GP)
        self.base_weights = nn.Parameter(torch.randn(out_channels, in_channels, filter_size, filter_size))
        
        # GP parameters for spatial correlation
        self.spatial_lengthscale = nn.Parameter(torch.tensor(1.0))
        self.spatial_variance = nn.Parameter(torch.tensor(0.1))
        
        # Learnable perturbation from GP
        self.gp_perturbation = nn.Parameter(torch.zeros(out_channels, in_channels, filter_size, filter_size))
        
    def get_spatial_prior_regularization(self):
        """
        Regularization based on spatial GP prior
        Encourages smooth filters
        """
        # Compute spatial distances between filter positions
        f = self.filter_size
        positions = torch.stack(torch.meshgrid(torch.arange(f), torch.arange(f), indexing='ij'), dim=-1).float()
        positions = positions.view(-1, 2).to(self.base_weights.device)
        
        # Compute distances
        dists = torch.cdist(positions, positions)
        
        # GP kernel
        K = self.spatial_variance * torch.exp(-0.5 * dists**2 / self.spatial_lengthscale**2)
        K = K + 1e-4 * torch.eye(K.shape[0], device=K.device)
        
        # Inverse kernel (precision)
        K_inv = torch.inverse(K)
        
        # Regularization: encourages perturbations to follow GP prior
        perturb_flat = self.gp_perturbation.view(self.out_channels, self.in_channels, -1)
        reg = torch.sum(perturb_flat @ K_inv @ perturb_flat.transpose(-1, -2))
        
        return reg
    
    def forward(self, x):
        """Apply GP-modulated convolution"""
        # Combine base weights with GP perturbation
        weights = self.base_weights + self.gp_perturbation
        
        out = F.conv2d(x, weights, padding=self.filter_size//2)
        return out

# ==================== Approach 3: Hybrid CNN + GP Refinement ====================
class GPRefinementLayer(nn.Module):
    """
    Takes CNN features and applies GP-based refinement
    Useful for few-shot adaptation
    """
    def __init__(self, num_features, num_inducing=50):
        super().__init__()
        self.num_features = num_features
        self.num_inducing = num_inducing
        
        # Inducing points (learned)
        self.inducing_points = nn.Parameter(torch.randn(num_inducing, num_features))
        
        # GP parameters
        self.lengthscale = nn.Parameter(torch.tensor(1.0))
        self.outputscale = nn.Parameter(torch.tensor(0.1))
        
        # Refinement weights (variational parameters)
        self.refinement_mean = nn.Parameter(torch.zeros(num_features))
        self.refinement_std = nn.Parameter(torch.ones(num_features) * 0.01)
        
    def rbf_kernel(self, x1, x2):
        """RBF kernel"""
        dist = torch.cdist(x1, x2)
        return self.outputscale * torch.exp(-0.5 * dist**2 / self.lengthscale**2)
    
    def forward(self, x):
        """
        Apply GP refinement to features
        x: (batch, num_features, height, width)
        """
        batch, c, h, w = x.shape
        
        # Spatially pool features
        x_pooled = F.adaptive_avg_pool2d(x, 1).squeeze(-1).squeeze(-1)  # (batch, num_features)
        
        # Compute kernel between inputs and inducing points
        K_xu = self.rbf_kernel(x_pooled, self.inducing_points)  # (batch, num_inducing)
        K_uu = self.rbf_kernel(self.inducing_points, self.inducing_points)  # (num_inducing, num_inducing)
        K_uu = K_uu + 1e-4 * torch.eye(K_uu.shape[0], device=K_uu.device)
        
        # GP prediction
        K_uu_inv = torch.inverse(K_uu)
        refinement = K_xu @ K_uu_inv @ self.refinement_mean.unsqueeze(1)  # (batch, 1)
        
        # Apply refinement
        refinement = refinement.view(batch, 1, 1, 1)
        refined_features = x * (1 + refinement)
        
        return refined_features

# ==================== Example Model Using GP Filters ====================
class GPConvNet(nn.Module):
    """
    Simple CNN using GP-generated filters
    """
    def __init__(self, num_classes=10, approach='generator'):
        super().__init__()
        
        if approach == 'generator':
            # Approach 1: GP Filter Generator
            self.conv1 = GPFilterGenerator(num_filters=16, filter_size=3)
            self.conv2 = GPFilterGenerator(num_filters=32, filter_size=3)
            
        elif approach == 'spatial':
            # Approach 2: GP Spatial Conv
            self.conv1 = GPSpatialConv(1, 16, filter_size=3)
            self.conv2 = GPSpatialConv(16, 32, filter_size=3)
            
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, num_classes)
        self.approach = approach
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
    
    def get_gp_regularization(self):
        """Get GP-based regularization"""
        if self.approach == 'spatial':
            return self.conv1.get_spatial_prior_regularization() + \
                   self.conv2.get_spatial_prior_regularization()
        return 0.0

# ==================== Visualization ====================
def visualize_gp_filters(model, save_path='gp_filters.png'):
    """Visualize learned GP filters"""
    if isinstance(model.conv1, GPFilterGenerator):
        filters = model.conv1.get_filters().detach().cpu()
        num_filters = min(16, filters.shape[0])
        
        fig, axes = plt.subplots(4, 4, figsize=(10, 10))
        for i, ax in enumerate(axes.flat):
            if i < num_filters:
                filter_img = filters[i, 0].numpy()
                ax.imshow(filter_img, cmap='viridis')
                ax.set_title(f'Filter {i+1}')
                ax.axis('off')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Filters visualized and saved to {save_path}")

# ==================== Demo ====================
def demo():
    print("="*70)
    print("Gaussian Process Convolutional Filters Demo")
    print("="*70)
    
    # Create dummy data
    print("\nCreating dummy image data...")
    batch_size = 8
    x = torch.randn(batch_size, 1, 28, 28).to(device)
    y = torch.randint(0, 10, (batch_size,)).to(device)
    
    # Test Approach 1: GP Filter Generator
    print("\n1. GP Filter Generator:")
    print("-" * 70)
    model1 = GPConvNet(approach='generator').to(device)
    
    print("Sampling filters from GP prior...")
    model1.conv1.sample_from_gp_prior()
    model1.conv2.sample_from_gp_prior()
    
    output1 = model1(x)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output1.shape}")
    print(f"Lengthscale: {model1.conv1.lengthscale.item():.4f}")
    
    # Test Approach 2: GP Spatial Conv
    print("\n2. GP Spatial Convolutional Layer:")
    print("-" * 70)
    model2 = GPConvNet(approach='spatial').to(device)
    output2 = model2(x)
    reg = model2.get_gp_regularization()
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output2.shape}")
    print(f"GP regularization: {reg.item():.4f}")
    print(f"Spatial lengthscale: {model2.conv1.spatial_lengthscale.item():.4f}")
    
    # Test Approach 3: GP Refinement
    print("\n3. GP Refinement Layer:")
    print("-" * 70)
    features = torch.randn(batch_size, 64, 7, 7).to(device)
    refinement_layer = GPRefinementLayer(num_features=64, num_inducing=32).to(device)
    refined = refinement_layer(features)
    
    print(f"Input features shape: {features.shape}")
    print(f"Refined features shape: {refined.shape}")
    print(f"Refinement magnitude: {(refined - features).abs().mean().item():.4f}")
    
    print("\n" + "="*70)
    print("Key Ideas for GP Convolutional Filters:")
    print("="*70)
    print("✓ Approach 1 (Generator): Sample filters from GP over spatial coords")
    print("  - Pro: Principled way to generate filters with spatial structure")
    print("  - Pro: Can control smoothness via lengthscale")
    print("  - Con: Fixed after sampling (not learned end-to-end)")
    print()
    print("✓ Approach 2 (Spatial Prior): Standard conv + GP regularization")
    print("  - Pro: Learned end-to-end with GP prior")
    print("  - Pro: Encourages spatially smooth filters")
    print("  - Con: More complex optimization")
    print()
    print("✓ Approach 3 (Refinement): Adapt pretrained features via GP")
    print("  - Pro: Great for few-shot learning")
    print("  - Pro: Captures uncertainty in adaptation")
    print("  - Con: Adds extra computation")
    print()
    print("="*70)
    print("Potential Applications:")
    print("- Few-shot image classification (like your EuroSAT task!)")
    print("- Uncertainty-aware conv filters")
    print("- Interpretable feature learning")
    print("- Transfer learning with limited data")
    print("="*70)

if __name__ == "__main__":
    demo()

Using device: cuda
Gaussian Process Convolutional Filters Demo

Creating dummy image data...

1. GP Filter Generator:
----------------------------------------------------------------------
Sampling filters from GP prior...


RuntimeError: Given groups=1, weight of size [16, 1, 3, 3], expected input[1, 8, 28, 28] to have 1 channels, but got 8 channels instead

In [ ]:
"""
Smart Attention Initialization: Making Attention "Pre-Tuned"

Core Question: Can we initialize attention mechanisms so intelligently that they 
work well out-of-the-box with minimal/no fine-tuning?

Key Insight: The problem with random initialization is it doesn't encode 
any prior knowledge about what "good" attention should look like.

Multiple Approaches Explored:
1. Task-Specific Priors (spatial locality, semantic similarity)
2. Meta-Learned Initialization (learn to initialize from many tasks)
3. GP-Based Initialization (sample from prior over attention patterns)
4. Symmetric/Structured Initialization (group theory, geometric priors)
5. Curriculum Initialization (start simple, gradually complex)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==================== Approach 1: Spatial Prior Initialization ====================
class SpatialPriorAttention(nn.Module):
    """
    Initialize attention with strong spatial locality bias
    Idea: Nearby tokens should attend to each other more
    Good for: Images, sequences with local structure
    """
    def __init__(self, dim, num_heads=8, spatial_temperature=1.0):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # Standard QKV projections
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Spatial prior parameters (learnable but well-initialized)
        self.spatial_temperature = nn.Parameter(torch.tensor(spatial_temperature))
        self.locality_bias = nn.Parameter(torch.zeros(1))  # How much to bias towards local
        
    def create_spatial_prior(self, seq_len, grid_size=None):
        """
        Create spatial distance-based prior
        For images: 2D spatial distance
        For sequences: 1D distance
        """
        if grid_size is not None:
            # 2D spatial prior (for images arranged as patches)
            h, w = grid_size
            y_coords = torch.arange(h, device=self.spatial_temperature.device).repeat_interleave(w)
            x_coords = torch.arange(w, device=self.spatial_temperature.device).repeat(h)
            
            coords = torch.stack([y_coords, x_coords], dim=1).float()  # (seq_len, 2)
            
            # Compute pairwise distances
            dist = torch.cdist(coords, coords)  # (seq_len, seq_len)
        else:
            # 1D sequential prior
            positions = torch.arange(seq_len, device=self.spatial_temperature.device).float()
            dist = torch.abs(positions.unsqueeze(1) - positions.unsqueeze(0))
        
        # Convert distance to similarity (closer = higher attention)
        # Use RBF-like kernel: exp(-dist^2 / temperature)
        spatial_prior = torch.exp(-dist**2 / (2 * self.spatial_temperature**2))
        
        return spatial_prior
    
    def forward(self, x, grid_size=None):
        """
        x: (batch, seq_len, dim)
        grid_size: (height, width) if input is image patches
        """
        B, N, C = x.shape
        
        # Standard attention computation
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Compute attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, N, N)
        
        # ADD SPATIAL PRIOR
        spatial_prior = self.create_spatial_prior(N, grid_size)  # (N, N)
        spatial_prior = spatial_prior.unsqueeze(0).unsqueeze(0)  # (1, 1, N, N)
        
        # Combine: learned attention + spatial prior
        attn = attn + self.locality_bias * torch.log(spatial_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 2: Meta-Learned Initialization ====================
class MetaLearnedAttention(nn.Module):
    """
    Learn the initialization from many related tasks (meta-learning)
    Idea: Find initialization that's good across many tasks with few updates
    Similar to MAML but for attention weights
    """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # Meta-learned initialization for QKV
        # These are "good starting points" learned from many tasks
        self.meta_qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Meta-learned attention pattern templates
        # These capture common attention patterns across tasks
        self.num_templates = 4
        self.attention_templates = nn.Parameter(torch.randn(self.num_templates, num_heads, 1, 1))
        self.template_weights = nn.Parameter(torch.ones(self.num_templates) / self.num_templates)
        
    def get_meta_attention_prior(self, seq_len):
        """
        Combine learned attention templates
        Each template represents a common pattern (e.g., local, global, skip)
        """
        # Create base patterns
        templates = []
        
        # Template 1: Local attention (diagonal band)
        local = torch.eye(seq_len, device=self.attention_templates.device)
        if seq_len > 1:
            local += torch.diag(torch.ones(seq_len - 1, device=self.attention_templates.device), diagonal=1)
            local += torch.diag(torch.ones(seq_len - 1, device=self.attention_templates.device), diagonal=-1)
        templates.append(local)
        
        # Template 2: Global attention (uniform)
        global_attn = torch.ones(seq_len, seq_len, device=self.attention_templates.device) / seq_len
        templates.append(global_attn)
        
        # Template 3: Skip attention (every other)
        skip = torch.zeros(seq_len, seq_len, device=self.attention_templates.device)
        skip[::2, ::2] = 1.0
        skip = skip / (skip.sum(dim=-1, keepdim=True) + 1e-8)
        templates.append(skip)
        
        # Template 4: Hierarchical (first token attends to all)
        hierarchical = torch.zeros(seq_len, seq_len, device=self.attention_templates.device)
        hierarchical[0, :] = 1.0 / seq_len  # First token = global
        hierarchical[1:, 1:] = torch.eye(seq_len - 1, device=self.attention_templates.device)  # Rest = local
        templates.append(hierarchical)
        
        # Combine templates with learned weights
        templates = torch.stack(templates)  # (num_templates, seq_len, seq_len)
        weights = F.softmax(self.template_weights, dim=0)
        
        combined = (templates * weights.view(-1, 1, 1)).sum(dim=0)
        
        return combined
    
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.meta_qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add meta-learned prior
        meta_prior = self.get_meta_attention_prior(N)  # (N, N)
        meta_prior = meta_prior.unsqueeze(0).unsqueeze(0)  # (1, 1, N, N)
        
        # Soft combination: learned + meta prior
        attn = attn + torch.log(meta_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 3: GP Prior on Attention ====================
class GPPriorAttention(nn.Module):
    """
    Sample attention patterns from Gaussian Process prior
    Idea: GP captures smooth attention patterns with uncertainty
    Good for: When you want structured but stochastic attention
    """
    def __init__(self, dim, num_heads=8, lengthscale=5.0):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # GP hyperparameters
        self.lengthscale = nn.Parameter(torch.tensor(lengthscale))
        self.outputscale = nn.Parameter(torch.tensor(1.0))
        
        # Cached GP samples (can resample during training)
        self.register_buffer('gp_attention_bias', None)
        
    def rbf_kernel(self, positions):
        """RBF kernel for position similarities"""
        dist = torch.cdist(positions, positions)
        K = self.outputscale * torch.exp(-0.5 * dist**2 / self.lengthscale**2)
        return K
    
    def sample_gp_attention_pattern(self, seq_len):
        """
        Sample attention bias from GP
        This gives structured randomness
        """
        positions = torch.arange(seq_len, device=self.lengthscale.device).float().unsqueeze(1)
        
        K = self.rbf_kernel(positions)  # (seq_len, seq_len)
        K = K + 1e-4 * torch.eye(seq_len, device=K.device)
        
        # Cholesky decomposition
        L = torch.linalg.cholesky(K)
        
        # Sample from GP: f = L @ z where z ~ N(0, I)
        z = torch.randn(seq_len, seq_len, device=K.device)
        gp_sample = L @ z
        
        return gp_sample
    
    def forward(self, x, resample_gp=False):
        B, N, C = x.shape
        
        # Sample GP pattern if needed
        if self.gp_attention_bias is None or resample_gp:
            self.gp_attention_bias = self.sample_gp_attention_pattern(N)
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add GP-sampled bias
        gp_bias = self.gp_attention_bias.unsqueeze(0).unsqueeze(0) * 0.1  # Scale down
        attn = attn + gp_bias
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 4: Symmetric/Geometric Initialization ====================
class SymmetricAttention(nn.Module):
    """
    Initialize with geometric/group-theoretic structure
    Idea: Exploit symmetries in the task (rotation, translation invariance)
    Good for: Images, graphs with known symmetries
    """
    def __init__(self, dim, num_heads=8, symmetry_type='rotational'):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.symmetry_type = symmetry_type
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Symmetry-preserving bias
        self.symmetry_strength = nn.Parameter(torch.tensor(1.0))
        
    def create_symmetric_prior(self, seq_len, grid_size=None):
        """
        Create attention pattern with specific symmetry
        """
        if self.symmetry_type == 'rotational' and grid_size is not None:
            # Rotational symmetry for 2D grids
            h, w = grid_size
            center = torch.tensor([h/2, w/2], device=self.symmetry_strength.device)
            
            # Create coordinate grid
            y = torch.arange(h, device=center.device).repeat_interleave(w)
            x = torch.arange(w, device=center.device).repeat(h)
            coords = torch.stack([y, x], dim=1).float()
            
            # Distance from center
            dist_from_center = torch.norm(coords - center, dim=1)
            
            # Attention decays with distance from center (rotationally symmetric)
            similarity = torch.exp(-dist_from_center.unsqueeze(1) * dist_from_center.unsqueeze(0) / 10.0)
            
        elif self.symmetry_type == 'translational':
            # Translation invariant: Toeplitz matrix
            positions = torch.arange(seq_len, device=self.symmetry_strength.device).float()
            relative_pos = positions.unsqueeze(1) - positions.unsqueeze(0)
            
            # Attention depends only on relative position
            similarity = torch.exp(-relative_pos.abs() / 5.0)
            
        else:
            # Default: permutation invariant (all equal)
            similarity = torch.ones(seq_len, seq_len, device=self.symmetry_strength.device) / seq_len
        
        return similarity
    
    def forward(self, x, grid_size=None):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add symmetric prior
        symmetric_prior = self.create_symmetric_prior(N, grid_size)
        symmetric_prior = symmetric_prior.unsqueeze(0).unsqueeze(0)
        
        attn = attn + self.symmetry_strength * torch.log(symmetric_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Approach 5: Curriculum Initialization ====================
class CurriculumAttention(nn.Module):
    """
    Start with simple attention (e.g., uniform), gradually allow complexity
    Idea: Like curriculum learning - start easy, get harder
    Good for: Training from scratch with small data
    """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Curriculum parameter: 0 = simple (uniform), 1 = complex (learned)
        self.complexity = nn.Parameter(torch.tensor(0.0))
        
    def get_curriculum_prior(self, seq_len):
        """
        Interpolate between simple and complex patterns
        """
        # Simple: uniform attention
        uniform = torch.ones(seq_len, seq_len, device=self.complexity.device) / seq_len
        
        # Medium: local attention
        local = torch.eye(seq_len, device=self.complexity.device)
        if seq_len > 1:
            local += 0.5 * torch.diag(torch.ones(seq_len - 1, device=self.complexity.device), diagonal=1)
            local += 0.5 * torch.diag(torch.ones(seq_len - 1, device=self.complexity.device), diagonal=-1)
        local = local / local.sum(dim=-1, keepdim=True)
        
        # Interpolate based on curriculum stage
        complexity = torch.sigmoid(self.complexity)
        
        if complexity < 0.5:
            # Stage 1: uniform -> local
            alpha = complexity * 2
            prior = (1 - alpha) * uniform + alpha * local
        else:
            # Stage 2: local -> learned (no prior)
            alpha = (complexity - 0.5) * 2
            prior = (1 - alpha) * local + alpha * torch.ones_like(local) / seq_len
        
        return prior
    
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add curriculum prior
        curriculum_prior = self.get_curriculum_prior(N)
        curriculum_prior = curriculum_prior.unsqueeze(0).unsqueeze(0)
        
        # Strength of prior decreases as we learn
        prior_strength = 5.0 * (1.0 - torch.sigmoid(self.complexity))
        attn = attn + prior_strength * torch.log(curriculum_prior + 1e-8)
        
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        
        return out, attn

# ==================== Visualization & Demo ====================
def visualize_attention_patterns():
    """Compare attention patterns from different initialization strategies"""
    seq_len = 16
    dim = 64
    batch = 1
    
    x = torch.randn(batch, seq_len, dim)
    
    models = {
        'Spatial Prior': SpatialPriorAttention(dim),
        'Meta-Learned': MetaLearnedAttention(dim),
        'GP Prior': GPPriorAttention(dim),
        'Symmetric': SymmetricAttention(dim, symmetry_type='translational'),
        'Curriculum': CurriculumAttention(dim),
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (name, model) in enumerate(models.items()):
        if idx < len(axes):
            with torch.no_grad():
                _, attn = model(x)
                
            # Average over heads and batch
            attn_avg = attn[0].mean(dim=0).numpy()  # (seq_len, seq_len)
            
            axes[idx].imshow(attn_avg, cmap='viridis', aspect='auto')
            axes[idx].set_title(f'{name}\nAttention Pattern')
            axes[idx].set_xlabel('Key Position')
            axes[idx].set_ylabel('Query Position')
            axes[idx].colorbar = plt.colorbar(axes[idx].images[0], ax=axes[idx])
    
    # Hide last subplot if odd number
    if len(models) < len(axes):
        axes[-1].axis('off')
    
    plt.tight_layout()
    plt.savefig('/tmp/attention_initialization_comparison.png', dpi=150, bbox_inches='tight')
    print("Visualization saved!")

def demo():
    print("="*70)
    print("Smart Attention Initialization Strategies")
    print("="*70)
    
    print("\n🎯 GOAL: Initialize attention so well it barely needs tuning!\n")
    
    seq_len = 16
    dim = 64
    x = torch.randn(1, seq_len, dim)
    
    print("="*70)
    print("1. SPATIAL PRIOR ATTENTION")
    print("="*70)
    print("Idea: Nearby tokens should attend to each other")
    print("Prior: exp(-distance^2 / temperature)")
    spatial = SpatialPriorAttention(dim)
    out1, attn1 = spatial(x, grid_size=(4, 4))
    print(f"✓ Output shape: {out1.shape}")
    print(f"✓ Spatial temperature: {spatial.spatial_temperature.item():.2f}")
    print(f"✓ Locality bias: {spatial.locality_bias.item():.2f}")
    
    print("\n" + "="*70)
    print("2. META-LEARNED INITIALIZATION")
    print("="*70)
    print("Idea: Learn initialization from many tasks (MAML-style)")
    print("Templates: Local, Global, Skip, Hierarchical")
    meta = MetaLearnedAttention(dim)
    out2, attn2 = meta(x)
    weights = F.softmax(meta.template_weights, dim=0)
    print(f"✓ Output shape: {out2.shape}")
    print(f"✓ Template weights: {weights.detach().numpy()}")
    
    print("\n" + "="*70)
    print("3. GP PRIOR ATTENTION")
    print("="*70)
    print("Idea: Sample structured attention from Gaussian Process")
    print("Kernel: RBF with learned lengthscale")
    gp = GPPriorAttention(dim)
    out3, attn3 = gp(x, resample_gp=True)
    print(f"✓ Output shape: {out3.shape}")
    print(f"✓ GP lengthscale: {gp.lengthscale.item():.2f}")
    print(f"✓ GP outputscale: {gp.outputscale.item():.2f}")
    
    print("\n" + "="*70)
    print("4. SYMMETRIC/GEOMETRIC ATTENTION")
    print("="*70)
    print("Idea: Exploit task symmetries (rotation, translation)")
    print("Prior: Distance from center (rotational symmetry)")
    symmetric = SymmetricAttention(dim, symmetry_type='translational')
    out4, attn4 = symmetric(x)
    print(f"✓ Output shape: {out4.shape}")
    print(f"✓ Symmetry type: {symmetric.symmetry_type}")
    print(f"✓ Symmetry strength: {symmetric.symmetry_strength.item():.2f}")
    
    print("\n" + "="*70)
    print("5. CURRICULUM ATTENTION")
    print("="*70)
    print("Idea: Start simple (uniform), gradually allow complexity")
    print("Stages: Uniform → Local → Learned")
    curriculum = CurriculumAttention(dim)
    out5, attn5 = curriculum(x)
    print(f"✓ Output shape: {out5.shape}")
    print(f"✓ Complexity stage: {torch.sigmoid(curriculum.complexity).item():.2f}")
    
    print("\n" + "="*70)
    print("KEY INSIGHTS")
    print("="*70)
    print("✓ Random init has NO inductive bias → needs lots of data")
    print("✓ Smart init encodes task structure → works with few samples")
    print("✓ Can combine approaches (e.g., Spatial + Meta-learned)")
    print("✓ For YOUR EuroSAT: Spatial prior perfect for satellite images!")
    print()
    print("NEXT STEPS:")
    print("→ Replace ViT attention with SpatialPriorAttention")
    print("→ Should work better with 20 samples (encodes locality bias)")
    print("→ Can still fine-tune, but starts from better place")
    print("="*70)

if __name__ == "__main__":
    demo()
    # visualize_attention_patterns()